In [ ]:
%pip install rl4co

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve() / 'src'))

from dvrptw_bench.data.instance_filters import find_rc_instances
from dvrptw_bench.data.solomon_parser import parse_solomon
from dvrptw_bench.heuristics.ortools_solver import ORToolsVRPTWSolver
from dvrptw_bench.heuristics.gls import GLSSolver
from dvrptw_bench.dynamic.simulator import DynamicSimulator
from dvrptw_bench.viz.route_plot import plot_routes
from dvrptw_bench.viz.timeline_plot import plot_timeline
from dvrptw_bench.viz.convergence_plot import plot_convergence
from dvrptw_bench.viz.inspector import inspect_dynamic, inspect_static 
from dvrptw_bench.rl.rl4co_runner import RL4COPolicy 

import logging
import sys

gls_logger = logging.getLogger("dvrptw_bench.heuristics.gls")
gls_logger.handlers.clear()
out = Path('../outputs/notebook_heuristics')

handler = logging.StreamHandler(sys.stdout)  # send to notebook cell output
handler.setLevel(logging.DEBUG)
handler.setFormatter(logging.Formatter("%(asctime)s| %(levelname)s | %(message)s"))

gls_logger.addHandler(handler)
gls_logger.setLevel(logging.DEBUG)
gls_logger.propagate = False

In [ ]:
dataset_root = Path('../dataset')
instances = find_rc_instances(dataset_root)
assert instances, 'Place RC*.txt files under ../dataset/solomon_rc100/'
instance = parse_solomon(instances[0])
instance.instance_id, instance.n_customers

## Ortools

In [ ]:
ortools_solver = ORToolsVRPTWSolver()
solution = ortools_solver.solve(instance, 5)
solution.total_distance

## RL

In [ ]:
"""RL4CO model builder wrappers."""

from __future__ import annotations

from pathlib import Path

import rl4co
import torch
from rl4co.envs import CVRPTWEnv, CVRPEnv
from rl4co.models.rl import REINFORCE
from rl4co.models.zoo.am import AttentionModelPolicy
from rl4co.utils.trainer import RL4COTrainer
from tensordict import TensorDict



class RLModel:
    def __init__(self, device=None, env=None, policy=None, model=None, max_epochs=100):
        self.device = device
        self.model = model
        self.policy = policy
        self.env = env
        self.max_epochs = max_epochs
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # RL4CO env based on TorchRL
        if env is None:
            self.env = CVRPTWEnv(generator_params={"num_loc": 100,})

        # Policy: neural network, in this case with encoder-decoder architecture
        if policy is None:
            self.policy = AttentionModelPolicy(env_name=self.env.name).to(self.device)

        # RL Model: REINFORCE and greedy rollout baseline
        if model is None:
            self.model = REINFORCE(
                self.env,
                self.policy,
                baseline="rollout",
                batch_size=512,
                train_data_size=100_000,
                val_data_size=10_000,
                optimizer_kwargs={"lr": 1e-4},
            )
        self.trainer = RL4COTrainer(
            max_epochs=max_epochs,
            accelerator="gpu",
            devices=1,
            logger=None,
        )

    def train(self):
        self.trainer.fit(self.model)

    def save(self, path: str | Path) -> None:
        self.trainer.save_checkpoint(path)

    def load(self, path: str | Path) -> None:
        torch.serialization.safe_globals([rl4co.envs.routing.cvrp.env.CVRPEnv])
        self.model = self.model.load_from_checkpoint(path, weights_only=False, load_baseline=False)
        self.model.load_from_checkpoint(path, weights_only=False, load_baseline=False)
def build_attention_model():
    try:
        return RLModel()
    except Exception:
        return None
